### Persona-Conditioned Abstractive Summarization Engine

This notebook implements the **Controlled Text Generation & Abstractive Summarization** module for the University Communications Intelligence pipeline.

It fine-tunes a sequence-to-sequence transformer model (`facebook/bart-base`) to transform lengthy institutional circulars into concise, role-tailored summaries for specific target audiences (**Students**, **Faculty**, and **Administrators**).

---

### Key Features & Pipeline Capabilities
* Task-Conditioned Generation: role-prefixed prompts (`"summarize for student | category: ... | dates: ... | notice: ..."`) control summary scope and tone. Category and extracted deadlines are placed *before* the notice text so they survive truncation.
* Deterministic Audience Routing: personas that are not in `target_audiences` are never sent to the model. They return `"No specific [persona] action stated."` directly.
* URL / Email Masking: URLs and emails are replaced with `[URL1]`, `[EMAIL1]` placeholders before training and generation, then swapped back from the Stage 2 entity lists. The model can never invent a link.
* Leak-Free Evaluation: train / eval split is done per notice (not per row), so the same circular never appears in both splits.
* Evaluation & Guardrail Framework:
  * ROUGE Metrics (with a lead-words baseline for comparison).
  * Grounding Guardrail: flags numbers, links and emails in a summary that are not present in the source notice.

In [5]:
import hashlib
import json
import random
import re
from pathlib import Path

import pandas as pd
from datasets import Dataset

EVAL_FRACTION = 0.15
SEED = 42
ROLES = ["student", "faculty", "admin"]
DATA_PATH = r"submission\week_8\data\stage3_combined_pipeline_fmt.jsonl"


def find_data_file(path_str):
    relative_path = Path(str(path_str).replace("\\", "/"))
    if relative_path.is_absolute():
        data_file = relative_path if relative_path.is_file() else None
    else:
        candidates = [base / relative_path for base in [Path.cwd(), *Path.cwd().parents]]
        data_file = next((path for path in candidates if path.is_file()), None)
    if data_file is None:
        raise FileNotFoundError(f"Could not find {path_str!r}. Current directory: {Path.cwd()}")
    return data_file


def load_jsonl(path):
    records = []
    with path.open(encoding="utf-8") as file:
        for line_number, line in enumerate(file, start=1):
            if line.strip():
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError as error:
                    raise ValueError(f"Invalid JSON in {path} at line {line_number}") from error
    return records


def clean_whitespace(text):
    return re.sub(r"\s+", " ", text).strip()


def mask_entities(text, urls, emails):
    masked = text
    for index, url in sorted(enumerate(urls), key=lambda pair: -len(pair[1])):
        masked = masked.replace(url, f"[URL{index + 1}]")
    for index, email in enumerate(emails):
        masked = re.sub(re.escape(email), f"[EMAIL{index + 1}]", masked, flags=re.IGNORECASE)
    return re.sub(r"https?://[^\s<>\"')\]]+", "[LINK]", masked)


def resolve_placeholders(summary, urls, emails):
    def replace_url(match):
        index = int(match.group(1)) - 1
        return urls[index] if 0 <= index < len(urls) else ""

    def replace_email(match):
        index = int(match.group(1)) - 1
        return emails[index] if 0 <= index < len(emails) else ""

    summary = re.sub(r"\[URL(\d+)\]", replace_url, summary)
    summary = re.sub(r"\[EMAIL(\d+)\]", replace_email, summary)
    return clean_whitespace(summary.replace("[LINK]", ""))


def build_model_input(role, category, dates, notice_text):
    date_part = "; ".join(dates[:5]) if dates else "none"
    return f"summarize for {role} | category: {category} | dates: {date_part} | notice: {notice_text}"


def notice_group_key(input_text):
    notice = input_text.split("| notice: ", 1)[-1]
    return hashlib.sha256(clean_whitespace(notice).casefold().encode("utf-8")).hexdigest()


DATA_FILE = find_data_file(DATA_PATH)
records = load_jsonl(DATA_FILE)
all_rows = []
seen_pairs = set()

for record in records:
    input_text = clean_whitespace(record["input_text"])
    target_summary = clean_whitespace(record["target_summary"])
    match = re.match(r"^summarize for (student|faculty|admin)\b", input_text, re.IGNORECASE)
    if match is None:
        raise ValueError(f"Could not determine persona for {record.get('doc_id')}")
    if not target_summary:
        raise ValueError(f"Empty target summary for {record.get('doc_id')}")

    pair_key = (input_text, target_summary)
    if pair_key in seen_pairs:
        continue
    seen_pairs.add(pair_key)

    placeholder_text = input_text + " " + target_summary
    url_count = max(map(int, re.findall(r"\[URL(\d+)\]", placeholder_text)), default=0)
    email_count = max(map(int, re.findall(r"\[EMAIL(\d+)\]", placeholder_text)), default=0)
    urls = [f"[URL{index}]" for index in range(1, url_count + 1)]
    emails = [f"[EMAIL{index}]" for index in range(1, email_count + 1)]
    all_rows.append({
        "input_text": input_text,
        "target_summary": target_summary,
        "reference_summary": target_summary,
        "role": match.group(1).lower(),
        "doc_id": record["doc_id"],
        "urls_json": json.dumps(urls),
        "emails_json": json.dumps(emails),
        "group_key": notice_group_key(input_text),
    })

if not all_rows:
    raise ValueError(f"No training examples found in {DATA_FILE}")

print(f"Loaded {len(all_rows)} unique summary pairs from {DATA_FILE}")
print("Rows per role:")
print(pd.DataFrame(all_rows)["role"].value_counts().to_string())

doc_to_rows = {}
for row in all_rows:
    doc_to_rows.setdefault(row["group_key"], []).append(row)

role_to_docs = {role: [] for role in ROLES}
for group_key, rows in doc_to_rows.items():
    roles = {row["role"] for row in rows}
    primary_role = "admin" if "admin" in roles else "faculty" if "faculty" in roles else "student"
    role_to_docs[primary_role].append(group_key)

eval_keys = set()
for role, keys in role_to_docs.items():
    random.Random(SEED).shuffle(keys)
    num_eval = min(max(1, int(len(keys) * EVAL_FRACTION)), max(0, len(keys) - 1))
    eval_keys.update(keys[:num_eval])

train_rows = []
eval_rows = []
for group_key, rows in doc_to_rows.items():
    destination = eval_rows if group_key in eval_keys else train_rows
    destination.extend({key: value for key, value in row.items() if key != "group_key"} for row in rows)

if not train_rows or not eval_rows:
    raise ValueError("Training and evaluation splits must both contain examples")
assert not ({row["doc_id"] for row in train_rows} & {row["doc_id"] for row in eval_rows}), "Notice leakage between train and eval splits"

raw_dataset = Dataset.from_list(train_rows)
eval_dataset = Dataset.from_list(eval_rows)
print(f"Train rows: {len(raw_dataset)} | Eval rows: {len(eval_dataset)}")
print(f"Eval roles: {pd.Series(eval_dataset['role']).value_counts().to_dict()}")

Loaded 188 unique summary pairs from c:\ProjectFiles\uni-comms-intelligence\submission\week_8\data\stage3_combined_pipeline_fmt.jsonl
Rows per role:
role
student    112
faculty     68
admin        8
Train rows: 161 | Eval rows: 27
Eval roles: {'student': 17, 'faculty': 9, 'admin': 1}


In [6]:
import math
import numpy as np
import torch
import evaluate
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer,
    set_seed,
)

set_seed(SEED)

model_name = "facebook/bart-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

MAX_INPUT_LENGTH = 1024
MAX_TARGET_LENGTH = 160
WEEK8_MODELS_DIR = DATA_FILE.parents[1] / "models"
SAVE_DIR = str(WEEK8_MODELS_DIR / "summarizer_final")
CHECKPOINT_DIR = WEEK8_MODELS_DIR / "summarizer_checkpoints"

# Used by the Trainer during evaluation and by the final generation function
model.generation_config.max_length = MAX_TARGET_LENGTH
model.generation_config.num_beams = 4


def preprocess_function(examples):
    # No padding here: the collator pads each batch dynamically and masks label padding with -100
    model_inputs = tokenizer(examples["input_text"], max_length=MAX_INPUT_LENGTH, truncation=True)
    labels = tokenizer(text_target=examples["target_summary"], max_length=MAX_TARGET_LENGTH, truncation=True)
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


input_lengths = [len(tokenizer(text)["input_ids"]) for text in raw_dataset["input_text"]]
target_lengths = [len(tokenizer(text_target=text)["input_ids"]) for text in raw_dataset["target_summary"]]

print(f"Train inputs longer than {MAX_INPUT_LENGTH} tokens (will be truncated): "
      f"{sum(length > MAX_INPUT_LENGTH for length in input_lengths)} / {len(input_lengths)}")
print(f"Train targets longer than {MAX_TARGET_LENGTH} tokens (will be truncated): "
      f"{sum(length > MAX_TARGET_LENGTH for length in target_lengths)} / {len(target_lengths)}")
print(f"Median input tokens: {int(np.median(input_lengths))} | Median target tokens: {int(np.median(target_lengths))}")

tokenized_dataset = raw_dataset.map(preprocess_function, batched=True, remove_columns=raw_dataset.column_names)
tokenized_eval_dataset = eval_dataset.map(preprocess_function, batched=True, remove_columns=eval_dataset.column_names)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8,
)

rouge = evaluate.load("rouge")


def compute_metrics(eval_pred):
    predictions, labels = eval_pred

    if isinstance(predictions, tuple):
        predictions = predictions[0]

    predictions = np.where(predictions != -100, predictions, tokenizer.pad_token_id)
    labels = np.where(labels != -100, labels, tokenizer.pad_token_id)

    decoded_predictions = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)

    scores = rouge.compute(
        predictions=decoded_predictions,
        references=decoded_labels,
        use_stemmer=True,
    )

    result = {key: float(value) for key, value in scores.items()}
    result["gen_len"] = float(np.mean([np.count_nonzero(ids != tokenizer.pad_token_id) for ids in predictions]))

    return result


num_epochs = 5
per_device_batch_size = 4
gradient_accumulation_steps = 2

steps_per_epoch = max(1, math.ceil(len(tokenized_dataset) / (per_device_batch_size * gradient_accumulation_steps)))
total_steps = steps_per_epoch * num_epochs
warmup_steps = int(0.1 * total_steps)

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
use_fp16 = torch.cuda.is_available() and not use_bf16

training_args = Seq2SeqTrainingArguments(
    output_dir=str(CHECKPOINT_DIR),
    num_train_epochs=num_epochs,
    per_device_train_batch_size=per_device_batch_size,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=gradient_accumulation_steps,
    learning_rate=3e-5,                  
    weight_decay=0.01,
    warmup_steps=warmup_steps,
    label_smoothing_factor=0.1,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,                  
    load_best_model_at_end=True,
    metric_for_best_model="rougeL",
    greater_is_better=True,
    predict_with_generate=True,
    logging_steps=5,
    bf16=use_bf16,
    fp16=use_fp16,
    report_to="none",
)
from transformers import EarlyStoppingCallback
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    eval_dataset=tokenized_eval_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]  
)

print("Starting Fine-Tuning Loop...")
trainer.train()
print("Training Complete!")

print(f"Best checkpoint: {trainer.state.best_model_checkpoint}")

trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print(f"Saved best model to {SAVE_DIR}")

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

Train inputs longer than 1024 tokens (will be truncated): 7 / 161
Train targets longer than 160 tokens (will be truncated): 0 / 161
Median input tokens: 204 | Median target tokens: 52


Map:   0%|          | 0/161 [00:00<?, ? examples/s]

Map:   0%|          | 0/27 [00:00<?, ? examples/s]

Starting Fine-Tuning Loop...


...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

[transformers] There were missing keys in the checkpoint model loaded: ['model.encoder.embed_tokens.weight', 'model.decoder.embed_tokens.weight', 'lm_head.weight'].


Training Complete!
Best checkpoint: c:\ProjectFiles\uni-comms-intelligence\submission\week_8\models\summarizer_checkpoints\checkpoint-105


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Saved best model to c:\ProjectFiles\uni-comms-intelligence\submission\week_8\models\summarizer_final


In [7]:
# To reuse the trained model later without retraining:
# tokenizer = AutoTokenizer.from_pretrained(SAVE_DIR)
# model = AutoModelForSeq2SeqLM.from_pretrained(SAVE_DIR)


def generate_from_input(input_text: str) -> str:
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=MAX_INPUT_LENGTH,
    ).to(model.device)

    model.eval()

    with torch.no_grad():
        summary_ids = model.generate(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            max_new_tokens=150,
            min_new_tokens=15,
            num_beams=4,
            no_repeat_ngram_size=6,
            early_stopping=True,
            generation_config=None,
        )

    return tokenizer.decode(summary_ids[0], skip_special_tokens=True)


def generate_persona_summary(
    raw_notice: str,
    role: str = "student",
    category: str = "announcements",
    dates: list = None,
    urls: list = None,
    emails: list = None,
    target_audiences: list = None,
) -> str:
    # AUDIENCE ROUTING RULE: never ask the model to write for a persona that is not targeted
    if target_audiences is not None and role not in target_audiences:
        return f"No specific {role} action stated."

    dates = dates or []
    urls = urls or []
    emails = emails or []

    masked_notice = clean_whitespace(mask_entities(raw_notice, urls, emails))
    input_text = build_model_input(role, category, dates, masked_notice)

    raw_summary = generate_from_input(input_text)

    # FACTUAL GUARDRAIL: URLs / emails only ever come from the Stage 2 entity lists
    return resolve_placeholders(raw_summary, urls, emails)


# Test User Inputs (use the same fields Stage 2 gives you: text, category, dates, urls, emails, audiences)
test_notice = (
    "Reminder: Last date for payment of Semester I tuition fees. Dear Students, This is a reminder that "
    "the last date for payment of tuition fees for Semester I, AY 2025-26, is 20.07.2025. Payments can be "
    "made through the student portal or by demand draft at the Accounts counter between 10:00 am and 3:00 pm. "
    "A late fee of Rs. 500 per day will be charged after the due date. For fee-related queries, write to "
    "accounts.support@example.edu or visit Room 105, Admin Block."
)

test_kwargs = dict(
    category="academic",
    dates=["20.07.2025"],
    emails=["accounts.support@example.edu"],
    target_audiences=["student", "faculty"],
)

print("\n--- USER GENERATION TEST ---")
print("Student View:", generate_persona_summary(test_notice, role="student", **test_kwargs))
print("Faculty View:", generate_persona_summary(test_notice, role="faculty", **test_kwargs))
print("Admin View:  ", generate_persona_summary(test_notice, role="admin", **test_kwargs))


--- USER GENERATION TEST ---


[transformers] Both `max_new_tokens` (=150) and `max_length`(=160) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=15) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=160) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=15) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/do

Student View: Semester I tuition fees for AY 2025-26 are due 20.07.2025. Payments can be made through the student portal or by demand draft at the Accounts counter between 10:00 am and 3:00 pm. Late fee of Rs. 500 per day is charged after the due date.
Faculty View: Semester I tuition fees for AY 2025-26 are due 20.07.2025. Payments can be made through the student portal or by demand draft at the Accounts counter between 10:00 am and 3:00 pm. Late fee of Rs. 500 per day is charged after the due date.
Admin View:   No specific admin action stated.


In [8]:
# 1. ROUGE Metrics Evaluation on the held-out eval split (notices the model never saw)
predictions = []
references = []
baseline_predictions = []
grounding_reports = []

for row in eval_dataset:
    urls = json.loads(row["urls_json"])
    emails = json.loads(row["emails_json"])

    raw_summary = generate_from_input(row["input_text"])

    predictions.append(resolve_placeholders(raw_summary, urls, emails))
    references.append(row["reference_summary"])

    # Baseline: just the first 60 words of the notice. The model should clearly beat this.
    notice_part = row["input_text"].split("| notice: ", 1)[-1]
    baseline_text = " ".join(notice_part.split()[:60])
    baseline_predictions.append(resolve_placeholders(baseline_text, urls, emails))

    grounding_reports.append((row["input_text"], raw_summary, urls, emails))

model_scores = rouge.compute(predictions=predictions, references=references, use_stemmer=True)
baseline_scores = rouge.compute(predictions=baseline_predictions, references=references, use_stemmer=True)

print("\n--- EVALUATION RESULTS ---")
print("Model    ROUGE:", {key: round(float(value), 4) for key, value in model_scores.items()})
print("Baseline ROUGE:", {key: round(float(value), 4) for key, value in baseline_scores.items()})


# 2. Grounding Check (numbers, links and emails must exist in the source)
def digit_groups(text):
    return {group.lstrip("0") or "0" for group in re.findall(r"\d+", text)}


def check_grounding(input_text: str, raw_summary: str, urls: list, emails: list, verbose: bool = False) -> bool:
    summary_without_placeholders = re.sub(r"\[(?:URL|EMAIL)\d+\]", "", raw_summary)

    ungrounded_numbers = sorted(digit_groups(summary_without_placeholders) - digit_groups(input_text))

    bad_links = [
        link for link in re.findall(r"https?://\S+", raw_summary)
        if link.rstrip(".,;)") not in urls
    ]

    bad_emails = [
        email for email in re.findall(r"[\w.+-]+@[\w-]+\.[\w.]+", raw_summary)
        if email.rstrip(".") not in emails
    ]

    bad_placeholders = []
    for kind, pool in [("URL", urls), ("EMAIL", emails)]:
        for number in re.findall(rf"\[{kind}(\d+)\]", raw_summary):
            if not 1 <= int(number) <= len(pool):
                bad_placeholders.append(f"[{kind}{number}]")

    has_hallucination = bool(ungrounded_numbers or bad_links or bad_emails or bad_placeholders)

    if verbose:
        if has_hallucination:
            print(
                "Hallucination Warning!",
                {"numbers": ungrounded_numbers, "links": bad_links, "emails": bad_emails, "placeholders": bad_placeholders},
            )
        else:
            print("Grounding Check Passed: no ungrounded numbers, links or emails detected.")

    return has_hallucination


flags = [check_grounding(*report) for report in grounding_reports]
print(f"\nSummaries with ungrounded content: {sum(flags)} / {len(flags)}")
print("(Numbers written in a different format than the source, e.g. '25th July' vs '25.07', can be flagged as false positives.)")


# 3. Side-by-side look at a few eval examples
print("\n--- SAMPLE OUTPUTS ---")
for index in range(min(3, len(predictions))):
    print(f"\n[{eval_dataset[index]['role']}] {eval_dataset[index]['doc_id']}")
    print("Reference :", references[index])
    print("Generated :", predictions[index])
    check_grounding(*grounding_reports[index], verbose=True)

[transformers] Both `max_new_tokens` (=150) and `max_length`(=160) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=15) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=150) and `max_length`(=160) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `min_new_tokens` (=15) and `min_length`(=0) seem to have been set. `min_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/do


--- EVALUATION RESULTS ---
Model    ROUGE: {'rouge1': 0.5659, 'rouge2': 0.3504, 'rougeL': 0.4769, 'rougeLsum': 0.476}
Baseline ROUGE: {'rouge1': 0.4292, 'rouge2': 0.2401, 'rougeL': 0.3339, 'rougeLsum': 0.3352}

Summaries with ungrounded content: 1 / 27
(Numbers written in a different format than the source, e.g. '25th July' vs '25.07', can be flagged as false positives.)

--- SAMPLE OUTPUTS ---

[student] acad-002-user-manual-photocopy-url-on-nmims-grievance-redre__student
Reference : The user manual for the Photocopy URL and Re-valuation module of the NMIMS Grievance Redressal online application has been shared; refer to the attached manual before applying.
Generated : The User Manual for Photocopy URL and re-valuation of the NMIMS Grievance redressal online application is attached.
Grounding Check Passed: no ungrounded numbers, links or emails detected.

[student] acad-014-attendance-shortage-semester-iv__student
Reference : Attendance of several Semester IV students is below 75% as